In [1]:
from pathlib import Path
from astropy.io import fits
from astropy.stats import sigma_clipped_stats

import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

In [2]:
# Directory containing your pbcor FITS files
final_dir = Path(
    "/scratch/gsc9zr/Machine-Learning-Project/"
    "Astromorph_project/alma_TAP/final_fits_file"
)

# Find all pbcor FITS files
pbcor_files = sorted(final_dir.glob("*pbcor*.fits"))

print("Total pbcor FITS files:", len(pbcor_files))

for f in pbcor_files:
    print(f.name)

Total pbcor FITS files: 21
member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a3.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.pbcor.fits
member.uid___A001_X133d_X3b81.W42-MME_sci.spw25.cube.I.pbcor.fits
member.uid___A001_X1465_X1bc.Lh01_sci.spw29.cube.I.pbcor.fits
member.uid___A001_X1465_X1bc.Lh03_sci.spw29.cube.I.pbcor.fits
member.uid___A001_X1465_X1c0.S01_sci.spw29.cube.I.pbcor.fits
member.uid___A001_X1465_X1c0.S02_sci.spw29.cube.I.pbcor.fits
member.uid___A001_X1465_X1c0.S03_sci.spw29.cube.I.pbcor.fits
member.uid___A001_X1465_X1c0.S04_sci.spw29.cube.I.pbcor.fits
member.uid___A001_X1465_X1c0.S05_sci.spw29.cube.I.pbcor.fits
member.uid___A001_X1465_X1c0.S06_sci.spw29.cube.I.pbcor.fits
member.uid___A001_X1465_X1c0.S07_sci.spw29.cube.I.pbcor.fits
member.uid___A001_X1465_X1c0.S08_sci.spw29.cube.I.pbcor.fits
member.uid___A001_X1465_X1c0.S09_sci.spw29.cube.I.pbcor.fits
member.uid___A001_X1465_X1c0.S10_sci.spw29.cube.I.pb

In [3]:
# Dictionary to store all FITS cubes
fits_cubes = {}

# Loop through all pbcor FITS files
for fits_file in pbcor_files:

    # Extract source name
    source_name = fits_file.name.split(".")[2].split("_sci")[0]

    # Open FITS cube
    with fits.open(fits_file, memmap=True) as hdul:
        data = np.squeeze(hdul[0].data)
        header = hdul[0].header.copy()

    # Check dimensions
    if data.ndim != 3:
        print("Skipping:", fits_file.name)
        continue

    nchan, ny, nx = data.shape

    # Store cube and header
    fits_cubes[fits_file.name] = {
        "source_name": source_name,
        "data": data,
        "header": header
    }

    print("Source:", source_name)
    print("Cube shape:", data.shape)
    print("Velocity channels:", nchan)
    print("Y pixels:", ny)
    print("X pixels:", nx)
    print("-------------------------")

print("Total 3D FITS cubes loaded:", len(fits_cubes))

Source: ari_l
Cube shape: (958, 1050, 1050)
Velocity channels: 958
Y pixels: 1050
X pixels: 1050
-------------------------
Source: ari_l
Cube shape: (958, 240, 240)
Velocity channels: 958
Y pixels: 240
X pixels: 240
-------------------------
Source: W42-MME
Cube shape: (1918, 588, 588)
Velocity channels: 1918
Y pixels: 588
X pixels: 588
-------------------------
Source: Lh01
Cube shape: (3837, 480, 480)
Velocity channels: 3837
Y pixels: 480
X pixels: 480
-------------------------
Source: Lh03
Cube shape: (3837, 480, 480)
Velocity channels: 3837
Y pixels: 480
X pixels: 480
-------------------------
Source: S01
Cube shape: (3836, 450, 450)
Velocity channels: 3836
Y pixels: 450
X pixels: 450
-------------------------
Source: S02
Cube shape: (3833, 450, 450)
Velocity channels: 3833
Y pixels: 450
X pixels: 450
-------------------------
Source: S03
Cube shape: (3834, 450, 450)
Velocity channels: 3834
Y pixels: 450
X pixels: 450
-------------------------
Source: S04
Cube shape: (3834, 450, 45

In [4]:


# Dictionary to store RMS values
rms_values = {}

# Loop through all FITS cubes
for filename, cube_info in fits_cubes.items():

    source_name = cube_info["source_name"]
    data = cube_info["data"]

    # Get cube dimensions
    nchan, ny, nx = data.shape

    # Select central region
    y1, y2 = ny // 4, 3 * ny // 4
    x1, x2 = nx // 4, 3 * nx // 4

    # Sample every 5th channel and spatial pixel
    noise_data = data[::5, y1:y2:5, x1:x2:5]

    # Remove NaN and infinite values
    noise_data = noise_data[np.isfinite(noise_data)]

    if noise_data.size == 0:
        print("Skipping:", source_name, "- No valid data")
        continue

    # Calculate RMS estimate
    mean, median, std = sigma_clipped_stats(
        noise_data,
        sigma=3,
        maxiters=3
    )

    rms = std

    # Store RMS for each FITS file
    rms_values[filename] = rms

    print("Source:", source_name)
    print("Estimated RMS:", rms)
    print("-------------------------")

print("Total RMS values calculated:", len(rms_values))

Source: ari_l
Estimated RMS: 0.020735629
-------------------------
Source: ari_l
Estimated RMS: 0.020941595
-------------------------
Source: W42-MME
Estimated RMS: 0.0015725787
-------------------------
Source: Lh01
Estimated RMS: 0.007559516
-------------------------
Source: Lh03
Estimated RMS: 0.0077487133
-------------------------
Source: S01
Estimated RMS: 0.0044984743
-------------------------
Source: S02
Estimated RMS: 0.0045592925
-------------------------
Source: S03
Estimated RMS: 0.0044046813
-------------------------
Source: S04
Estimated RMS: 0.0043102675
-------------------------
Source: S05
Estimated RMS: 0.0045408006
-------------------------
Source: S06
Estimated RMS: 0.0045080315
-------------------------
Source: S07
Estimated RMS: 0.0043432633
-------------------------
Source: S08
Estimated RMS: 0.0043213437
-------------------------
Source: S09
Estimated RMS: 0.004526848
-------------------------
Source: S10
Estimated RMS: 0.0045593604
-------------------------
Sour

In [ ]:
from scipy.ndimage import zoom

# Desired spatial dimensions
new_ny = 128
new_nx = 128

# Loop through all FITS cubes
for filename, cube_info in fits_cubes.items():

    source_name = cube_info["source_name"]
    data = cube_info["data"]

    # Original cube dimensions
    nchan, ny, nx = data.shape

    print("Source:", source_name)
    print("Original cube shape:", data.shape)

    # Calculate resizing factors
    zoom_factors = (
        1,
        new_ny / ny,
        new_nx / nx
    )

    # Resize the 3D cube
    resized_data = zoom(
        data,
        zoom_factors,
        order=1
    )

    # Store resized cube
    fits_cubes[filename]["resized_data"] = resized_data

    print("Resized cube shape:", resized_data.shape)
    print("-------------------------")

print("Total FITS cubes resized:", len(fits_cubes))

Source: ari_l
Original cube shape: (958, 1050, 1050)
Resized cube shape: (958, 128, 128)
-------------------------
Source: ari_l
Original cube shape: (958, 240, 240)
Resized cube shape: (958, 128, 128)
-------------------------
Source: W42-MME
Original cube shape: (1918, 588, 588)
Resized cube shape: (1918, 128, 128)
-------------------------
Source: Lh01
Original cube shape: (3837, 480, 480)
Resized cube shape: (3837, 128, 128)
-------------------------
Source: Lh03
Original cube shape: (3837, 480, 480)
Resized cube shape: (3837, 128, 128)
-------------------------
Source: S01
Original cube shape: (3836, 450, 450)
Resized cube shape: (3836, 128, 128)
-------------------------
Source: S02
Original cube shape: (3833, 450, 450)
Resized cube shape: (3833, 128, 128)
-------------------------
Source: S03
Original cube shape: (3834, 450, 450)


In [ ]:


# Loop through all FITS cubes
for filename, cube_info in fits_cubes.items():

    source_name = cube_info["source_name"]

    # Original and resized cubes
    data = cube_info["data"]
    resized_data = cube_info["resized_data"]

    # Original cube dimensions
    nchan, ny, nx = data.shape

    # Original moment 8 map
    original_moment8 = np.nanmax(data, axis=0)

    # Resized moment 8 map
    resized_moment8 = np.nanmax(resized_data, axis=0)

    # Plot side by side
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Use same intensity scale
    vmin = np.nanpercentile(original_moment8, 5)
    vmax = np.nanpercentile(original_moment8, 99)

    # Original image
    im1 = axes[0].imshow(
        original_moment8,
        origin="lower",
        cmap="inferno",
        vmin=vmin,
        vmax=vmax
    )

    axes[0].set_title(
        f"{source_name} - Original ({ny} x {nx})"
    )

    axes[0].set_xlabel("X Pixel")
    axes[0].set_ylabel("Y Pixel")

    # Resized image
    im2 = axes[1].imshow(
        resized_moment8,
        origin="lower",
        cmap="inferno",
        vmin=vmin,
        vmax=vmax
    )

    axes[1].set_title(
        f"{source_name} - Resized (128 x 128)"
    )

    axes[1].set_xlabel("X Pixel")
    axes[1].set_ylabel("Y Pixel")

    # Colorbars
    plt.colorbar(im1, ax=axes[0], label="Peak Intensity")
    plt.colorbar(im2, ax=axes[1], label="Peak Intensity")

    plt.tight_layout()
    plt.show()

    plt.close(fig)

In [ ]:
# Dictionary to store scaled coordinates
dbscan_inputs = {}

# Define scale parameters
spatial_scale = 2.0   # pixels
spectral_scale = 2.0  # channels

# Loop through all FITS cubes
for filename, cube_info in fits_cubes.items():

    source_name = cube_info["source_name"]

    # Get emission coordinates for this cube
    x = emission_data[filename]["x"]
    y = emission_data[filename]["y"]
    z = emission_data[filename]["z"]

    # Spatial coordinates
    spatial_coords = np.column_stack((x, y)).astype(float)

    # Spectral coordinates
    spectral_coords = z.reshape(-1, 1).astype(float)

    # Scale coordinates
    X = np.column_stack((
        spatial_coords / spatial_scale,
        spectral_coords / spectral_scale
    ))

    # Store DBSCAN input for each FITS file
    dbscan_inputs[filename] = X

    print("Source:", source_name)
    print("DBSCAN input shape:", X.shape)
    print("-------------------------")

print("Total FITS cubes prepared:", len(dbscan_inputs))

In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np

# Dictionary to store DBSCAN results
dbscan_results = {}

# Loop through all FITS cubes
for filename, cube_info in fits_cubes.items():

    source_name = cube_info["source_name"]

    # Get scaled coordinates for this cube
    X = dbscan_inputs[filename]

    print("Processing:", source_name)

    # Check if there are any emission voxels
    if len(X) == 0:
        print("No significant emission detected. Skipping.")
        print("-------------------------")
        continue

    # Initialize DBSCAN
    dbscan = DBSCAN(
        eps=1.5,
        min_samples=10
    )

    # Fit DBSCAN
    labels = dbscan.fit_predict(X)

    # Number of clusters
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

    # Number of noise voxels
    n_noise = np.sum(labels == -1)

    # Store results for each FITS file
    dbscan_results[filename] = {
        "source_name": source_name,
        "labels": labels,
        "n_clusters": n_clusters,
        "n_noise": n_noise
    }

    print("Source:", source_name)
    print("Number of clusters:", n_clusters)
    print("Number of noise voxels:", n_noise)
    print("-------------------------")

print("Total FITS cubes analyzed:", len(dbscan_results))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create 5x5 grid for 21 FITS cubes
fig, axes = plt.subplots(5, 5, figsize=(20, 20))

axes = axes.flatten()

# Loop through all FITS cubes
for i, (filename, result) in enumerate(dbscan_results.items()):

    ax = axes[i]

    source_name = result["source_name"]
    labels = result["labels"]

    # Get coordinates for this FITS cube
    x = emission_data[filename]["x"]
    y = emission_data[filename]["y"]

    # Exclude DBSCAN noise
    valid = labels >= 0

    # Plot clusters
    if np.any(valid):

        sc = ax.scatter(
            x[valid],
            y[valid],
            c=labels[valid],
            cmap="tab20",
            s=2
        )

    ax.set_xlabel("X Pixel")
    ax.set_ylabel("Y Pixel")

    ax.set_title(
        f"{source_name} - {result['n_clusters']} clusters",
        fontsize=10
    )

    # Use resized cube dimensions
    ny, nx = fits_cubes[filename]["resized_data"].shape[1:]

    ax.set_xlim(-0.5, nx - 0.5)
    ax.set_ylim(-0.5, ny - 0.5)
    ax.set_aspect("equal")

# Hide unused panels
for j in range(len(dbscan_results), len(axes)):
    axes[j].axis("off")

plt.suptitle(
    "3D DBSCAN Clusters - ALMA SiO Outflows",
    fontsize=18
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create 7x6 grid (21 sources, 2 panels per source)
fig, axes = plt.subplots(7, 6, figsize=(24, 28))

# Loop through all FITS cubes
for i, (filename, result) in enumerate(dbscan_results.items()):

    source_name = result["source_name"]
    labels = result["labels"]

    # Use resized cube
    data = fits_cubes[filename]["resized_data"]

    # Get cube dimensions
    nchan, ny, nx = data.shape

    # Get emission coordinates
    x = emission_data[filename]["x"]
    y = emission_data[filename]["y"]

    # Create moment 8 (peak intensity) map
    moment8 = np.full((ny, nx), np.nan)

    finite_pixels = np.isfinite(data).any(axis=0)

    moment8[finite_pixels] = np.max(
        np.where(np.isfinite(data), data, -np.inf),
        axis=0
    )[finite_pixels]

    # Determine row and column
    row = i // 3
    col = (i % 3) * 2

    # Left panel: Moment 8 map
    ax1 = axes[row, col]

    im = ax1.imshow(
        moment8,
        origin="lower",
        cmap="inferno"
    )

    ax1.set_title(f"{source_name} - Moment 8", fontsize=10)
    ax1.set_xlabel("X Pixel")
    ax1.set_ylabel("Y Pixel")

    # Right panel: DBSCAN clusters
    ax2 = axes[row, col + 1]

    valid = labels >= 0

    if np.any(valid):
        sc = ax2.scatter(
            x[valid],
            y[valid],
            c=labels[valid],
            cmap="tab20",
            s=2
        )

    ax2.set_xlim(-0.5, nx - 0.5)
    ax2.set_ylim(-0.5, ny - 0.5)
    ax2.set_aspect("equal")

    ax2.set_title(
        f"{source_name} - {result['n_clusters']} Clusters",
        fontsize=10
    )

    ax2.set_xlabel("X Pixel")
    ax2.set_ylabel("Y Pixel")

# Add overall title
fig.suptitle(
    "ALMA SiO Moment 8 Maps vs 3D DBSCAN Clusters",
    fontsize=20
)

plt.tight_layout(rect=[0, 0, 1, 0.98])

# Save figure
plt.savefig(
    final_dir / "Moment8_vs_DBSCAN_21_sources.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np

# Dictionary to store cluster statistics
cluster_statistics = {}

# Loop through all FITS cubes
for filename, result in dbscan_results.items():

    source_name = result["source_name"]
    labels = result["labels"]

    # Get cluster IDs (excluding noise)
    cluster_ids = sorted(set(labels) - {-1})

    print("\nSource:", source_name)
    print("File:", filename)
    print("Cluster statistics:")
    print("-------------------")

    # Store statistics for this FITS file
    cluster_statistics[filename] = {}

    for cluster_id in cluster_ids:

        cluster_mask = labels == cluster_id

        n_voxels = np.sum(cluster_mask)

        # Store number of voxels
        cluster_statistics[filename][cluster_id] = n_voxels

        print(
            f"Cluster {cluster_id}: "
            f"{n_voxels} voxels"
        )

    print("Total clusters:", len(cluster_ids))
    print("-------------------------")

print("\nTotal FITS cubes analyzed:", len(cluster_statistics))